### **Dev Log - 6 Maret 2026**

**Bagian**  
Data Preprocessing – Pendetailan Data Duplikasi (Sebelum dan Sesudah Noise Removal)

**Aktivitas**  
Melakukan pendetailan data duplikasi pada dua tahap preprocessing, yaitu sebelum dan setelah *noise removal*. Pendetailan dilakukan dengan menghitung frekuensi kemunculan setiap teks menggunakan metode `value_counts()`, kemudian menampilkan contoh data duplikat beserta jumlah kemunculannya.

Hasil dari masing-masing tahap juga disimpan ke dalam file CSV untuk memudahkan analisis lanjutan.

**Hasil**  

1. **Sebelum Noise Removal**  
   - Duplikasi yang terdeteksi relatif sedikit  
   - Frekuensi kemunculan didominasi oleh nilai rendah (2–5 kali)  
   - Variasi penulisan masih tinggi sehingga banyak teks serupa belum terdeteksi sebagai duplikasi  

2. **Setelah Noise Removal**  
   - Jumlah duplikasi meningkat secara signifikan  
   - Ditemukan teks dengan frekuensi kemunculan tinggi (hingga puluhan kali)  
   - Pola duplikasi menjadi lebih jelas setelah teks dinormalisasi  

**Insight**  
Perbedaan hasil antara sebelum dan setelah *noise removal* menunjukkan bahwa proses pembersihan data berperan penting dalam mengurangi variasi penulisan.

Normalisasi teks menyebabkan data yang sebelumnya terlihat berbeda menjadi identik, sehingga meningkatkan deteksi duplikasi.

**Output**  
- Menampilkan contoh data duplikat beserta jumlah kemunculannya pada masing-masing tahap  
- Menyimpan hasil ke dalam file:  
  `validasidata/detailduplikat1.csv` (sebelum noise removal)  
  `validasidata/detailduplikat2.csv` (setelah noise removal)

**Catatan**  
Pendetailan ini digunakan untuk menganalisis pengaruh *noise removal* terhadap pola duplikasi data, serta sebagai dasar perbandingan dengan skenario preprocessing lainnya.

### **Dev Log - 8 Maret 2026**

**Bagian**  
Data Preprocessing – Penyesuaian Urutan Data Cleaning

**Aktivitas**  
Melakukan penyesuaian urutan tahapan *data cleaning* berdasarkan saran dari dosen pembimbing, dengan mengubah alur preprocessing menjadi:
- *Noise removal* dilakukan di awal  
- Dilanjutkan dengan pengecekan *missing values*  
- Kemudian proses *data deduplication*  

Selain itu, dibuat notebook baru untuk mengimplementasikan urutan tersebut tanpa menghapus hasil dari struktur sebelumnya, guna membandingkan hasil dari kedua skenario.

**Catatan**  
Penyesuaian ini bertujuan untuk mengevaluasi pengaruh urutan preprocessing terhadap hasil pembersihan data, khususnya dalam mendeteksi duplikasi.

### **Dev Log - 17 Maret 2026**

**Bagian**
Data Preprocessing – Data Cleaning (Noise Removal)

**Perubahan**
Melakukan revisi pada proses *noise removal*, khususnya pada penanganan tanda baca.

Sebelumnya, seluruh tanda baca dihapus. Namun, dilakukan perbaikan dengan mempertahankan tanda tanya (?) dan tanda seru (!) dalam teks.

**Alasan Perubahan**
Penyesuaian ini dilakukan berdasarkan pendekatan heuristik pada metode, yang mempertimbangkan tanda baca sebagai bagian dari penilaian sentimen.

Berdasarkan referensi dari implementasi kode pada repositori GitHub pengembang VADER:
- Tanda seru (!) memiliki kontribusi terhadap intensitas sentimen, jika digunakan berulang (maksimal hingga 4 tanda seru).
- Tanda tanya (?) juga memengaruhi interpretasi sentimen, ketika digunakan secara berulang (sekitar 2–3 kali).

**Dampak**
Dengan mempertahankan kedua tanda baca tersebut, diharapkan model dapat menangkap nuansa emosi dalam teks secara lebih akurat pada tahap analisis sentimen.

**Catatan**
Perubahan ini akan berpengaruh pada hasil akhir analisis sentimen, sehingga perlu dilakukan evaluasi ulang pada tahap berikutnya.

# **DATA CLEANING**
Pada data cleaning terdapat 3 tahapan, yaitu:
1. Checking Missing Values
2. Data Deduplication
3. Noise Removal

In [1]:
# 1.1 Import Library
import pandas as pd
import os
import re

In [2]:
# 1.2 Load Dataset Hasil Data Selection
df = pd.read_csv('dataselection/dataset_selected.csv')

print("Data berhasil dimuat!")
print("Total data awal:", len(df), "baris")

df.head()

Data berhasil dimuat!
Total data awal: 16205 baris


,timestamp,text
0,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik & negar...
1,2016-12-30T06:30:36.000Z,"Tertibkan Media Online, DPR: Pemerintah Jangan..."
2,2016-12-30T04:48:35.000Z,@Portal_Kemlu_RI @DPR_RI @jokowi harus dievalu...
3,2016-12-30T04:21:40.000Z,"jangan ngambang, aturan logis apa undang-undan..."
4,2016-12-30T02:36:13.000Z,6. Kebebasan bersuara & berpendapat memang dij...


### CHECKING MISSING VALUES

In [3]:
# 1.3 Ringkasan Missing Values (NaN)

missing_before = df.isnull().sum()

print("Jumlah missing value sebelum cleaning:")
print(missing_before)

Jumlah missing value sebelum cleaning:
timestamp    0
text         0
dtype: int64


In [4]:
# 1.4 Checking Empty String / Blank Text

# Menghitung text kosong atau hanya berisi spasi
empty_text_count = df['text'].astype(str).str.strip().eq('').sum()

print("Jumlah text kosong (string kosong / hanya spasi):", empty_text_count)

Jumlah text kosong (string kosong / hanya spasi): 0


In [5]:
# 1.5 Simpan Checkpoint Setelah Missing Values

output_path = 'datacleaning'
os.makedirs(output_path, exist_ok=True)

df.to_csv(
    os.path.join(output_path, 'data_after_missing_values.csv'),
    index=False
)

print("Checkpoint data_after_missing_values.csv berhasil disimpan.")

Checkpoint data_after_missing_values.csv berhasil disimpan.


### DATA DEDUPLICATION


In [6]:
# 2.1 Load Data Hasil Missing Values
df = pd.read_csv('datacleaning/data_after_missing_values.csv')

print("Data hasil missing values berhasil dimuat.")
print("Total data sebelum deduplication:", len(df))

Data hasil missing values berhasil dimuat.
Total data sebelum deduplication: 16205


In [7]:
# 2.2 Ringkasan Duplikasi (Before Deduplication)

jumlah_data_sebelum = len(df)

# Menghitung jumlah baris yang terdeteksi sebagai duplikat
jumlah_duplikat = df.duplicated(subset=['text']).sum()

print("Jumlah data sebelum deduplication:", jumlah_data_sebelum)
print("Jumlah data duplikat terdeteksi:", jumlah_duplikat)

Jumlah data sebelum deduplication: 16205
Jumlah data duplikat terdeteksi: 16


In [8]:
# 2.3 Menampilkan Contoh Data Duplikat

df_duplikat_before = df[
    df.duplicated(subset=['text'], keep=False)
]

print("Total baris yang termasuk kelompok duplikat:", len(df_duplikat_before))
df_duplikat_before[['text']].head(10)

Total baris yang termasuk kelompok duplikat: 29


,text
24,Demikian ulasan singkat tentang Masa Reses DPR...
25,Masa Reses ini diharapkan dapat dimanfaatkan d...
26,Yg perlu diingat & kita pahami bersama bahwa D...
27,"Berdasarkan Tatib DPR Pasal 211 ayat 6, hasil ..."
28,Kegiatan reses para Anggota DPR dilakukan seca...
29,"Pada Masa Persidangan, aspirasi & masukan masy..."
31,Masa Reses sendiri tertuang dalam UU MD3 No.17...
32,Masa Reses bukan berarti para Anggota DPR tida...
33,"Dalam Masa Reses, Anggota DPR akan melakukan k..."
34,Masa Reses adalah masa dimana Anggota DPR mela...


In [9]:
#2.4 Pendetailan teks yang duplikat serta jumlah kemunculanya

import os

# Pastikan folder ada
folder_path = "validasidata"
os.makedirs(folder_path, exist_ok=True)

# Hitung jumlah kemunculan tiap teks
duplicate_counts = df['text'].value_counts()

# Ambil hanya yang duplikat (>1)
duplicate_counts = duplicate_counts[duplicate_counts > 1]

# Ubah ke DataFrame
duplicate_detail = duplicate_counts.reset_index()
duplicate_detail.columns = ['text', 'jumlah_kemunculan']

# Preview 10 data teratas
print("Contoh 10 data duplikat beserta jumlah kemunculannya:")
print(duplicate_detail.head(10))

# Simpan ke CSV
file_path = os.path.join(folder_path, "detailduplikat.csv")
duplicate_detail.to_csv(file_path, index=False)

print(f"\nData lengkap berhasil disimpan di: {file_path}")

Contoh 10 data duplikat beserta jumlah kemunculannya:
                                                text  jumlah_kemunculan
0  pembahasan rancangan undang-undang pemilihan u...                  5
1  Dalam Masa Reses, Anggota DPR akan melakukan k...                  2
2  Masa Reses adalah masa dimana Anggota DPR mela...                  2
3  Kegiatan reses para Anggota DPR dilakukan seca...                  2
4  Temuan Komisi IX DPR RI, Serbuan Buruh China A...                  2
5  Pada Masa Persidangan, aspirasi & masukan masy...                  2
6  #KTweets: Pembahasan RUU Penyandang Disabilita...                  2
7  Masa Reses sendiri tertuang dalam UU MD3 No.17...                  2
8  Masa Reses bukan berarti para Anggota DPR tida...                  2
9  Demikian ulasan singkat tentang Masa Reses DPR...                  2

Data lengkap berhasil disimpan di: validasidata\detailduplikat.csv


In [10]:
# 2.5 Proses Deduplication

df = df.drop_duplicates(
    subset=['text'],
    keep='first'
)

print("Proses deduplication selesai.")

Proses deduplication selesai.


In [11]:
# 2.5 Perbandingan Jumlah Data Before–After

jumlah_data_setelah = len(df)
jumlah_data_dihapus = jumlah_data_sebelum - jumlah_data_setelah

print("Jumlah data setelah deduplication:", jumlah_data_setelah)
print("Jumlah data yang dihapus:", jumlah_data_dihapus)

Jumlah data setelah deduplication: 16189
Jumlah data yang dihapus: 16


In [12]:
# 2.6 Verifikasi Duplikasi Setelah Deduplication

duplikat_setelah = df.duplicated(subset=['text']).sum()

print("Jumlah duplikat setelah deduplication:", duplikat_setelah)

Jumlah duplikat setelah deduplication: 0


In [13]:
# 2.7 Reset Index Setelah Deduplication

df = df.reset_index(drop=True)

print("Index berhasil direset.")

Index berhasil direset.


In [14]:
# 2.8 Simpan Checkpoint Data Setelah Deduplication

output_path = 'datacleaning'
os.makedirs(output_path, exist_ok=True)

df.to_csv(
    os.path.join(output_path, 'data_after_deduplication.csv'),
    index=False
)

print("Checkpoint data_after_deduplication.csv berhasil disimpan.")

Checkpoint data_after_deduplication.csv berhasil disimpan.


catatan:
data dimuat ulang untuk memastikan proses identifikasi duplikasi dilakukan pada data mentah sebelum dilakukan tahapan pembersihan dan pengolahan data, sehingga hasil yang diperoleh meemang menjelaskan kondisi asli dataset.

Hasil pemeriksaan data duplikasi menunjukkan bahwa terdapat 29 baris data yang terlibat dalam kemunculan teks berulang. Jumlah tersebut tidak selalu berbentuk pasangan duplikat, karena ditemukan satu narasi aspirasi yang dipublikasikan sebanyak lima kali oleh akun yang sama pada rentang tahun 2017 hingga 2021. Selain itu, ditemukan dua belas teks unik lain yang masing-masing muncul dua kali, sehingga dua belas baris tambahan dihapus. Dengan demikian, secara keseluruhan terdapat 16 baris data redundan yang dieliminasi untuk menjaga keadilan bobot setiap unit aspirasi dalam analisis sentimen

**total baris yang dihapus**
4 (dari @hamidari2) + 12 (dari pasangan)
= 16 baris

### NOISE REMOVAL

In [15]:
# 3.1 Import Library
import pandas as pd
import re
import os

In [16]:
# 3.2 Load Data Hasil Deduplication
df = pd.read_csv('datacleaning/data_after_deduplication.csv')

print("Data berhasil dimuat.")
print("Jumlah data sebelum noise removal:", len(df))

df.head()

Data berhasil dimuat.
Jumlah data sebelum noise removal: 16189


,timestamp,text
0,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik & negar...
1,2016-12-30T06:30:36.000Z,"Tertibkan Media Online, DPR: Pemerintah Jangan..."
2,2016-12-30T04:48:35.000Z,@Portal_Kemlu_RI @DPR_RI @jokowi harus dievalu...
3,2016-12-30T04:21:40.000Z,"jangan ngambang, aturan logis apa undang-undan..."
4,2016-12-30T02:36:13.000Z,6. Kebebasan bersuara & berpendapat memang dij...


In [17]:
# 3.3 Definisi Fungsi Noise Removal

def noise_removal(text):
    text = str(text)

    text = re.sub(r"http\S+|www\S+", "", text)   # hapus URL
    text = re.sub(r"@\w+", "", text)             # hapus mention
    text = re.sub(r"#(\w+)", r"\1", text)        # hapus simbol # tapi simpan katanya
    text = re.sub(r"&\w+;", "", text)            # hapus HTML entity
    text = re.sub(r"\d+", "", text)              # hapus angka
    text = re.sub(r"[^\w\s!?]", " ", text)       # hapus tanda baca kecuali ! dan ?
    text = re.sub(r"\s+", " ", text).strip()     # rapikan spasi

    return text

In [18]:
# 3.4 Menerapkan Noise Removal
df['text_clean'] = df['text'].apply(noise_removal)

df[['text', 'text_clean']].head()

,text,text_clean
0,ADIL loh utk yg punya kebijakan publik & negar...,ADIL loh utk yg punya kebijakan publik negara ...
1,"Tertibkan Media Online, DPR: Pemerintah Jangan...",Tertibkan Media Online DPR Pemerintah Jangan S...
2,@Portal_Kemlu_RI @DPR_RI @jokowi harus dievalu...,harus dievaluasi lg kebijakan bebas visa truta...
3,"jangan ngambang, aturan logis apa undang-undan...",jangan ngambang aturan logis apa undang undang
4,6. Kebebasan bersuara & berpendapat memang dij...,Kebebasan bersuara berpendapat memang dijamin ...


In [19]:
# Mengatur display pandas agar lebih rapi
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 100)  # Batasi lebar kolom saat preview

# Fungsi helper untuk memotong teks panjang 
def truncate_text(text, max_len=80):
    """Memotong teks menjadi max_len karakter + '...' jika terlalu panjang"""
    if pd.isna(text):
        return ""
    text = str(text)
    return text[:max_len] + "..." if len(text) > max_len else text

# 3.5 Filter Karakter Non-Latin
non_latin_mask = df['text_clean'].str.contains(
    r'[\uAC00-\uD7A3\u1100-\u11FF\u3130-\u318F'  # Hangul (Korea)
    r'\u4E00-\u9FFF\u3400-\u4DBF'                # CJK Unified dan Ext-A (Kanji/Hanzi)
    r'\u0400-\u04FF'                             # Kiril
    r'\u0370-\u03FF'                             # Yunani
    r'\u0E00-\u0E7F\u0900-\u097F]'               # Thai dan Devanagari
    , regex=True, na=False
)

count_non_latin = non_latin_mask.sum()
print(f"[VALIDASI] Ditemukan {count_non_latin} baris mengandung tulisan non-Latin.\n")

if count_non_latin > 0:
    print("[PREVIEW] Contoh data non-Latin yang akan dihapus:")
    print("-" * 120)
    
    # Ambil sampel, lalu buat kolom preview yang sudah dipotong
    preview_df = df.loc[non_latin_mask, ['text', 'text_clean']].head(3).copy()
    preview_df['text_preview'] = preview_df['text'].apply(truncate_text, max_len=70)
    preview_df['clean_preview'] = preview_df['text_clean'].apply(truncate_text, max_len=70)
    
    # Tampilkan hanya kolom preview yang sudah dipotong
    print(preview_df[['text_preview', 'clean_preview']].to_string(index=False))
    print("-" * 120 + "\n")
    
    # Menunjukkan karakter non-Latin apa yang terdeteksi
    import re
    print("[DETAIL] Karakter non-Latin yang terdeteksi:")
    for idx, row in df.loc[non_latin_mask, ['text_clean']].head(3).iterrows():
        non_latin_chars = re.findall(r'[\uAC00-\uD7A3\u4E00-\u9FFF\u0600-\u06FF\u0400-\u04FF]', str(row['text_clean']))
        if non_latin_chars:
            print(f"  • Baris {idx}: {list(set(non_latin_chars))}")
    print()
    
    # Eksekusi penghapusan
    df = df[~non_latin_mask].reset_index(drop=True)
    print(f"[INFO] {count_non_latin} baris berhasil dihapus.")

print(f"[STATUS] Jumlah data setelah filter non-Latin: {len(df):,}")

[VALIDASI] Ditemukan 1 baris mengandung tulisan non-Latin.

[PREVIEW] Contoh data non-Latin yang akan dihapus:
------------------------------------------------------------------------------------------------------------------------
                                                         text_preview                                                         clean_preview
국회 진입했던 계엄군들 철수 시작 계엄령 해제 국회의장 국회 진입 전원 찬성 계엄사령관 비상계엄 우리나라 공수부대 국회 본청 국회 진입했던 계엄군들 철수 시작 계엄령 해제 국회의장 국회 진입 전원 찬성 계엄사령관 비상계엄 우리나라 공수부대 국회 본청
------------------------------------------------------------------------------------------------------------------------

[DETAIL] Karakter non-Latin yang terdeteksi:
  • Baris 15916: ['장', '청', '관', '대', '비', '작', '찬', '진', '라', '시', '공', '엄', '했', '부', '의', '철', '원', '본', '리', '성', '입', '상', '해', '제', '수', '전', '령', '사', '나', '계', '국', '군', '던', '회', '우', '들']

[INFO] 1 baris berhasil dihapus.
[STATUS] Jumlah data setelah filter non-Latin: 16,188


In [22]:
# 3.6 Cek Teks Kosong Setelah Noise Removal

kosong_setelah_noise = df['text_clean'].str.strip().eq('').sum()

print("Jumlah text_clean kosong setelah noise removal:", kosong_setelah_noise)

# Hapus jika ada
df = df[df['text_clean'].str.strip() != '']

print("Jumlah data setelah hapus text_clean kosong:", len(df))

Jumlah text_clean kosong setelah noise removal: 0
Jumlah data setelah hapus text_clean kosong: 16166


In [23]:
# 3.7 Simpan Checkpoint Setelah Noise Removal

output_path = 'datacleaning'
os.makedirs(output_path, exist_ok=True)

df.to_csv(
    os.path.join(output_path, 'data_after_noise_removal.csv'),
    index=False
)

print("Checkpoint data_after_noise_removal.csv berhasil disimpan.")

Checkpoint data_after_noise_removal.csv berhasil disimpan.


# DATA DEDUPLICATION AFTER NOISE REMOVAL

Mengapa dilakukan penecekan duplikasi dan penanganan lagi setelah noise removal?
Hal tsb dikarenakan, peneliti menemukan kesamaan teks setelah dilakukan noise removal, kemungkinan terbesar penyebabnya adalah sudah dihapusnya karakter pada teks, sehingga hasilnya menjadi teks yang sama. 

Oleh karena itu dilakukan penanganan duplikasi data, agar data benar-benar bersih

In [24]:
# 4.1 Load Data After Noise Removal
df = pd.read_csv('datacleaning/data_after_noise_removal.csv')

print("Jumlah data sebelum dedup kedua:", len(df))

Jumlah data sebelum dedup kedua: 16166


In [25]:
# 4.2 Ringkasan Duplikasi Setelah Noise Removal

jumlah_duplikat = df.duplicated(subset=['text_clean']).sum()

print("Jumlah duplikat setelah noise removal:", jumlah_duplikat)

Jumlah duplikat setelah noise removal: 2974


In [26]:
# 4.3 Contoh Data Duplikat

df_duplikat = df[df.duplicated(subset=['text_clean'], keep=False)]

print("Total baris dalam kelompok duplikat:", len(df_duplikat))
df_duplikat[['text_clean']].head()

Total baris dalam kelompok duplikat: 3958


,text_clean
60,Anggota DPR Ini Minta Kebijakan Bebas Visa Dicabut
62,Anggota DPR Ini Minta Kebijakan Bebas Visa Dicabut
63,Anggota DPR Ini Minta Kebijakan Bebas Visa Dicabut
64,Anggota DPR Ini Minta Kebijakan Bebas Visa Dicabut
66,Anggota DPR Ini Minta Kebijakan Bebas Visa Dicabut


In [27]:
# 4.4 Pendetailan teks yang duplikat serta jumlah kemunculanya

import os

folder_path = "validasidata"
os.makedirs(folder_path, exist_ok=True)

# Hitung duplikat
duplicate_counts = df['text_clean'].value_counts()

# Ambil yang duplikat saja
duplicate_counts = duplicate_counts[duplicate_counts > 1]

print("Jumlah teks yang duplikat:", len(duplicate_counts))

# Ubah ke DataFrame (BIAR RAPI)
duplicate_detail = duplicate_counts.reset_index()
duplicate_detail.columns = ['text_clean', 'jumlah_kemunculan']

# Preview
print("\nContoh 10 data duplikat setelah noise removal:")
print(duplicate_detail.head(10))

# Simpan
file_path = os.path.join(folder_path, "detailduplikat1.csv")
duplicate_detail.to_csv(file_path, index=False)

print(f"\nData lengkap berhasil disimpan di: {file_path}")

Jumlah teks yang duplikat: 984

Contoh 10 data duplikat setelah noise removal:
                                                                               text_clean  jumlah_kemunculan
0                                       Henry Yosodiningrat Persen Anggota DPR Ikut Reses                 68
1  Di Ngawi Komisi X DPR RI Siapakan Delapan Item RUU Kebudayaan siaga indonesia Sindiran                 39
2                DPR RI Bersama Kemenpar RI Selenggarakan Bimtek Kebijakan Promosi Wisata                 31
3                                                  Anggota DPR RUU ITE mengacu putusan MK                 25
4                                           Anggota DPR ada pasal krusia dalam RUU Pemilu                 24
5                                       Anggota DPR pertanyakan dana repatriasi dalam RUU                 23
6                                      Anggota DPR Ini Minta Kebijakan Bebas Visa Dicabut                 23
7                                    Rapat Paripu

In [28]:
# 4.5 Proses Dedup Kedua

jumlah_sebelum = len(df)

df = df.drop_duplicates(subset=['text_clean'], keep='first')

jumlah_setelah = len(df)

print("Jumlah data setelah dedup kedua:", jumlah_setelah)
print("Total data dihapus pada tahap ini:", jumlah_sebelum - jumlah_setelah)

Jumlah data setelah dedup kedua: 13192
Total data dihapus pada tahap ini: 2974


In [29]:
# 4.6 Verifikasi Setelah Dedup Kedua

print("Sisa duplikat:",
      df.duplicated(subset=['text_clean']).sum())

Sisa duplikat: 0


In [30]:
# 4.7 Reset Index

df = df.reset_index(drop=True)

print("Index berhasil direset.")

Index berhasil direset.


## FORMATIING DATA DAN VALIDASI ISI

In [31]:
# 4.8 Formatting Final Dataset

# Rename kolom text_clean menjadi teks
df = df.rename(columns={'text_clean': 'teks'})

# Hapus kolom text lama
df = df.drop(columns=['text'], errors='ignore')

# Reset index dulu
df = df.reset_index(drop=True)

# Tambahkan nomor urut
df.insert(0, 'no', range(1, len(df) + 1))

# Atur ulang kolom
df = df[['no', 'timestamp', 'teks']]

print("Struktur akhir dataset:")
df.head()

Struktur akhir dataset:


,no,timestamp,teks
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara Ingat yg ini!!!
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan Sporadis Apalagi Selektif Hanya kepada Media yang B...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa trutama utk negara tiongkok pak!! bahaya tersembunyi ma...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin UU Namun kebebasan tersebut tdk harus kebablasan s...


In [32]:
# 4.9 Simpan Dataset Final Cleaning

output_path = 'datacleaning'
os.makedirs(output_path, exist_ok=True)

df.to_csv(
    os.path.join(output_path, 'data_cleaning_final.csv'),
    index=False
)

print("data_cleaning_final.csv berhasil disimpan.")

data_cleaning_final.csv berhasil disimpan.


In [33]:
print("Duplikat:", df.duplicated(subset=['teks']).sum())
print("NaN:", df['teks'].isna().sum())
print("Kosong:", (df['teks'].str.strip() == "").sum())
print("Total final:", len(df))

Duplikat: 0
NaN: 0
Kosong: 0
Total final: 13192
